In [1]:
import os
from sshtunnel import SSHTunnelForwarder
from pymongo import MongoClient
from dotenv import load_dotenv

# Load environment variables
load_dotenv("./.env")

def get_database_connection(SSH_HOST,
    SSH_PORT,
    SSH_USERNAME,
    SSH_PASSWORD,
    MONGO_HOST,
    MONGO_DB,
    LOCAL_BIND_PORT,
    REMOTE_BIND_PORT):
    
    # Establish SSH tunnel
    server = SSHTunnelForwarder(
        (SSH_HOST, SSH_PORT),  # Remote SSH server IP and SSH port
        ssh_username=SSH_USERNAME,
        ssh_password=SSH_PASSWORD,
        remote_bind_address=(MONGO_HOST, REMOTE_BIND_PORT),  # Remote MongoDB server and its port
        local_bind_address=('0.0.0.0', LOCAL_BIND_PORT)  # Local bind port
    )

    # Start the tunnel
    server.start()

    # Connect to the MongoDB server through the SSH tunnel
    client = MongoClient(host='127.0.0.1', port=server.local_bind_port)
    
    # Access the specific database
    db = client[MONGO_DB]
    
    return db, server

# Example usage with environment variables
SSH_HOST = "152.118.31.54"
SSH_PORT = 36022
SSH_USERNAME = "user01"
SSH_PASSWORD = "pass2024"
MONGO_HOST = "127.0.0.1"
MONGO_DB = "sispro-tews"
LOCAL_BIND_PORT = 23982  # Defaulting to 10022 if not provided
REMOTE_BIND_PORT = 23982  # Default MongoDB port

db, ssh_server = get_database_connection(SSH_HOST, SSH_PORT, SSH_USERNAME, SSH_PASSWORD, MONGO_HOST, MONGO_DB, LOCAL_BIND_PORT, REMOTE_BIND_PORT)

# After you're done with the DB connection, remember to stop the SSH tunnel
# ssh_server.stop() when you're finished using it.


In [2]:
db

Database(MongoClient(host=['127.0.0.1:23982'], document_class=dict, tz_aware=False, connect=True), 'sispro-tews')

In [3]:
station_datas = db["station"]
import pandas as pd

In [4]:
documents = list(station_datas.find())  # Convert cursor to list of documents

# Convert to Pandas DataFrame
df = pd.DataFrame(documents)

# Show the DataFrame (prints the first few rows)
print(df.head())

                        _id  \
0  6622587e2584cf1798833955   
1  6622587e2584cf1798833956   
2  6622587e2584cf1798833957   
3  6622587e2584cf1798833958   
4  6622587e2584cf1798833959   

                                                name  code network  \
0                  BBOO (Buckleboo, South Australia)  BBOO      AU   
1                   BCON (Beacon, Western Australia)  BCON      AU   
2                           CARL (Carlisle JUMP, WA)  CARL      AU   
3  CNB (Kowen Forrest, Australian Capital Territory)   CNB      AU   
4                     HTT (Hallett, South Australia)   HTT      AU   

           channel location   longitude   latitude  elevation server_seedlink  \
0  [BHE, BHN, BHZ]       00  136.058807 -32.810200      321.0  auspass.edu.au   
1  [BHE, BHN, BHZ]           117.764900 -30.335850      386.0  auspass.edu.au   
2  [BHE, BHN, BHZ]       00  115.926697 -31.983500        2.0  auspass.edu.au   
3  [BHE, BHN, BHZ]       00  149.362000 -35.312401      870.0  auspa

In [9]:
import paramiko
import subprocess
from kafka import KafkaProducer, KafkaConsumer

ssh_host = "152.118.31.54"
ssh_port = 36022
ssh_username = 'user01'
ssh_password = 'pass2024'
local_port = 9092  # Port on local machine to forward traffic to
remote_kafka_host = "152.118.31.54"
remote_kafka_port = 9999  # Kafka's port

# SSH command to open a tunnel
ssh_command = f'ssh -L {local_port}:{remote_kafka_host}:{remote_kafka_port} {ssh_username}@{ssh_host}'

# Open SSH tunnel using subprocess
process = subprocess.Popen(ssh_command, shell=True)

# Now connect to Kafka using 'localhost:9092'
from kafka import KafkaProducer

kafka_producer = KafkaProducer(bootstrap_servers=f'localhost:{local_port}')

# Send a message to Kafka
kafka_producer.send('testing_ssh', b'Message via SSH tunnel')
print("se")
# Close the producer connection
kafka_producer.close()

# Close SSH tunnel
process.terminate()

Pseudo-terminal will not be allocated because stdin is not a terminal.
